In [1]:
import sys
import os
import re
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path("../../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from lib.llm.llm_client import MODEL, MODEL_N, timed_completion, strip_thinking

In [2]:
# Cell 1 — Inspect folder structure
import os
from pathlib import Path

CFB_ROOT = PROJECT_ROOT / "documents" / "cfb"

for path in sorted(CFB_ROOT.rglob("*.pdf")):
    print(path.relative_to(CFB_ROOT))

CAC40/Axa/axa_climate_and_biodiversity_report_2024_va.pdf
CAC40/BNP Paribas/bnp_paribas_2023_climate_report.pdf
CAC40/Engie/Engie RI_Version EN_2105.pdf
CAC40/LVMH/015157_LVMH_RSE_committed_to_positive_impact_2023_GB_SR_MEL_080724 (6)_2024-09-19_10_56.pdf
CAC40/Orange/Orange 2023 IAR - On track.pdf
CAC40/Sanofi/Sanofi-GHU-Report-2024.pdf
CAC40/Suez/SUEZ_SD_Progress report 2023.pdf
CAC40/Total Energies S.A/totalenergies_sustainability-climate-2024-progress-report_2024_en_pdf (1).pdf
CAC40/Veolia Environment S.A/veolia-2024-climate-report.pdf
DAX/BASF/BASF_Report_2023.pdf
DAX/Siemens AG/2024-sustainability-report.pdf
Other/Ali Baba Group/2024 Alibaba Group Environmental, Social and Governance Report-0809.pdf
Other/ArcelorMittal SA/climate_action_report_2023.pdf
Other/BP/bp-sustainability-report-2023.pdf
Other/Baoshan Iron & Steel/2019.pdf
Other/Hindustan Unilever/hul-business-responsibility-sustainability-report-fy-2023-24.pdf
Other/NTPC/2017-2018-NTPC-Sustainability-Report-Final_opt.pdf

In [3]:
# Cell 2 — Setup output dirs
DATASETS_ROOT = PROJECT_ROOT / "datasets" / "experiments" / "agent" / "tasks"

DIVISION_DIR  = DATASETS_ROOT / "division"
EXTRACTION_DIR = DATASETS_ROOT / "extraction"
META_DIR      = DATASETS_ROOT / "meta"

for d in [DIVISION_DIR, EXTRACTION_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Dirs ready.")

Dirs ready.


In [4]:

import fitz  # pymupdf

def load_pdf_pages(pdf_path: Path) -> list[dict]:
    """Return list of {page_num, text} for each page."""
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if text:
            pages.append({"page_num": i + 1, "text": text})
    doc.close()
    return pages

# Inventory
pdf_files = sorted(CFB_ROOT.rglob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs")
for p in pdf_files:
    print(" ", p.relative_to(CFB_ROOT))

Found 33 PDFs
  CAC40/Axa/axa_climate_and_biodiversity_report_2024_va.pdf
  CAC40/BNP Paribas/bnp_paribas_2023_climate_report.pdf
  CAC40/Engie/Engie RI_Version EN_2105.pdf
  CAC40/LVMH/015157_LVMH_RSE_committed_to_positive_impact_2023_GB_SR_MEL_080724 (6)_2024-09-19_10_56.pdf
  CAC40/Orange/Orange 2023 IAR - On track.pdf
  CAC40/Sanofi/Sanofi-GHU-Report-2024.pdf
  CAC40/Suez/SUEZ_SD_Progress report 2023.pdf
  CAC40/Total Energies S.A/totalenergies_sustainability-climate-2024-progress-report_2024_en_pdf (1).pdf
  CAC40/Veolia Environment S.A/veolia-2024-climate-report.pdf
  DAX/BASF/BASF_Report_2023.pdf
  DAX/Siemens AG/2024-sustainability-report.pdf
  Other/Ali Baba Group/2024 Alibaba Group Environmental, Social and Governance Report-0809.pdf
  Other/ArcelorMittal SA/climate_action_report_2023.pdf
  Other/BP/bp-sustainability-report-2023.pdf
  Other/Baoshan Iron & Steel/2019.pdf
  Other/Hindustan Unilever/hul-business-responsibility-sustainability-report-fy-2023-24.pdf
  Other/NTPC/20

In [5]:

DIVISION_SYSTEM = """You are a document analyst. Given the full text of a PDF, decide the best splitting strategy.
Return ONLY valid JSON, no markdown, no explanation.
Schema:
{
  "strategy": "page | section | semantic | fixed",
  "reason": "short justification",
  "split_points": [list of page numbers or section titles where splits should occur]
}"""

def run_division_agent(pdf_path: Path, pages: list[dict]) -> dict:
    full_text = "\n\n".join([f"[Page {p['page_num']}]\n{p['text']}" for p in pages])
    messages = [
        {"role": "system", "content": DIVISION_SYSTEM},
        {"role": "user", "content": f"PDF: {pdf_path.name}\n\n{full_text[:12000]}"}
    ]
    response, latency_ms = timed_completion(messages, n=MODEL_N, extra_body={"chat_template_kwargs": {"enable_thinking": False}} if MODEL_N == 1 else {})
    raw = strip_thinking(response.choices[0].message.content)
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {"strategy": "error", "reason": raw, "split_points": []}
    return result, response, latency_ms

In [6]:

EXTRACTION_SYSTEM = """You are a financial document extractor. Given a text chunk from a PDF, extract the most meaningful content.
Return ONLY valid JSON, no markdown, no explanation.
Schema:
{
  "chunks": [
    {
      "title": "short label",
      "content": "extracted meaningful text",
      "type": "table | narrative | kpi | header | footnote | other",
      "relevance_score": 0.0-1.0
    }
  ]
}"""


def run_extraction_agent(chunk_text: str, source: str) -> dict:
    messages = [
        {"role": "system", "content": EXTRACTION_SYSTEM},
        {"role": "user", "content": f"Source: {source}\n\n{chunk_text[:8000]}"}
    ]
    response, latency_ms = timed_completion(messages, n=MODEL_N, extra_body={"chat_template_kwargs": {"enable_thinking": False}} if MODEL_N == 1 else {})
    raw = strip_thinking(response.choices[0].message.content)
    
    # Strip markdown fences if model wraps in ```json
    raw = re.sub(r"```json|```", "", raw).strip()
    
    print(f"RAW [{source}]: {repr(raw[:300])}")
    
    if not raw:
        print("EMPTY RESPONSE")
        return {"chunks": []}, response, latency_ms

    try:
        result = json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"JSON ERROR: {e}\nRAW: {raw[:500]}")
        result = {"chunks": []}
    return result, response, latency_ms

In [7]:
# Cell 6 — Chunk splitter helper
def split_pages(pages: list[dict], split_points: list) -> list[list[dict]]:
    """Split pages list at given page numbers."""
    if not split_points:
        return [pages]
    
    chunks, current = [], []
    for page in pages:
        if page["page_num"] in split_points and current:
            chunks.append(current)
            current = []
        current.append(page)
    if current:
        chunks.append(current)
    return chunks

In [8]:
# Cell 7 — Concurrent pipeline
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

division_records   = []
extraction_records = []
meta_records       = []

def process_pdf(pdf_path: Path) -> tuple:
    index_folder = pdf_path.relative_to(CFB_ROOT).parts[0]
    doc_name     = pdf_path.stem
    run_id       = f"{index_folder}__{doc_name}__{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    pages = load_pdf_pages(pdf_path)
    if not pages:
        return [], [], []

    # Division
    division_result, div_response, div_latency = run_division_agent(pdf_path, pages)

    div_row = {
        "run_id":       run_id,
        "index":        index_folder,
        "document":     doc_name,
        "strategy":     division_result.get("strategy"),
        "reason":       division_result.get("reason"),
        "split_points": json.dumps(division_result.get("split_points", [])),
        "input_tokens": div_response.usage.prompt_tokens,
        "output_tokens":div_response.usage.completion_tokens,
        "latency_ms":   div_latency,
        "model":        MODEL,
    }

    # Split
    split_points = [int(x) for x in division_result.get("split_points", []) if str(x).isdigit()]
    page_chunks  = split_pages(pages, split_points)

    # Extraction — concurrent per chunk
    ext_rows  = []
    ext_results_for_meta = []

    def extract_chunk(args):
        chunk_idx, chunk_pages = args
        chunk_text = "\n\n".join([f"[Page {p['page_num']}]\n{p['text']}" for p in chunk_pages])
        page_range = f"{chunk_pages[0]['page_num']}-{chunk_pages[-1]['page_num']}"
        ext_result, ext_response, ext_latency = run_extraction_agent(chunk_text, doc_name)
        rows = []
        for extracted in ext_result.get("chunks", []):
            rows.append({
                "run_id":          run_id,
                "index":           index_folder,
                "document":        doc_name,
                "chunk_idx":       chunk_idx,
                "page_range":      page_range,
                "original_chunk":  chunk_text,
                "title":           extracted.get("title"),
                "content":         extracted.get("content"),
                "type":            extracted.get("type"),
                "relevance_score": extracted.get("relevance_score"),
                "input_tokens":    ext_response.usage.prompt_tokens,
                "output_tokens":   ext_response.usage.completion_tokens,
                "latency_ms":      ext_latency,
                "model":           MODEL,
            })
        return rows, ext_result

    with ThreadPoolExecutor(max_workers=12) as chunk_executor:
        futures = {chunk_executor.submit(extract_chunk, (i, cp)): i for i, cp in enumerate(page_chunks)}
        for f in as_completed(futures):
            rows, ext_result = f.result()
            ext_rows.extend(rows)
            ext_results_for_meta.append(ext_result)

    meta_row = {
        "run_id":             run_id,
        "index":              index_folder,
        "document":           doc_name,
        "total_pages":        len(pages),
        "num_chunks":         len(page_chunks),
        "num_extractions":    sum(len(r.get("chunks", [])) for r in ext_results_for_meta),
        "division_tokens_in": div_response.usage.prompt_tokens,
        "division_tokens_out":div_response.usage.completion_tokens,
        "division_latency_ms":div_latency,
        "strategy":           division_result.get("strategy"),
        "timestamp":          datetime.now().isoformat(),
        "model":              MODEL,
        "model_n":            MODEL_N,
    }

    return div_row, ext_rows, meta_row


# Outer loop — concurrent per PDF
MAX_WORKERS = 6  # PDFs in parallel; tune based on API rate limits

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_pdf, p): p for p in pdf_files}
    for f in tqdm(as_completed(futures), total=len(pdf_files), desc="PDFs"):
        div_row, ext_rows, meta_row = f.result()
        if div_row:
            division_records.append(div_row)
            extraction_records.extend(ext_rows)
            meta_records.append(meta_row)

print(f"Done. Division: {len(division_records)} | Extractions: {len(extraction_records)} | Meta: {len(meta_records)}")

PDFs:   3%|▎         | 1/33 [00:38<20:33, 38.55s/it]

RAW [axa_climate_and_biodiversity_report_2024_va]: '{\n  "chunks": [\n    {\n      "title": "Report Title and Context",\n      "content": "2024 AXA Group Climate and Biodiversity Report: Roadmap to a Climate Transition Plan. Prepared in line with TCFD and TNFD recommendations to engage discussions on purpose, ambition, business readiness, and operating m'


PDFs:   6%|▌         | 2/33 [00:42<09:21, 18.12s/it]

RAW [bnp_paribas_2023_climate_report]: '{\n  "chunks": [\n    {\n      "title": "Report Overview",\n      "content": "BNP Paribas Climate Report (May 2024) covering Strategy, Risks & Opportunities, Net Zero Commitments, and Governance.",\n      "type": "header",\n      "relevance_score": 0.9\n    },\n    {\n      "title": "2025 Strategic Plan (GTS'


PDFs:   9%|▉         | 3/33 [00:44<05:19, 10.64s/it]

...

RAW [Pfizer_2023_Impact_Report_11MAR2024]: '{\n  "chunks": [\n    {\n      "title": "Report Overview and Purpose",\n      "content": "Pfizer is a global biopharmaceutical company focused on advancing its Purpose—breakthroughs that change patients\' lives. The ambition is to change a billion lives a year by 2027. This 2023 Impact Report covers the '


PDFs:  97%|█████████▋| 32/33 [08:22<01:05, 65.51s/it]

RAW [ge2022_sustainability_report]: '{\n  "chunks": [\n    {\n    "title": "Report Title and Theme",\n    "content": "2022 Sustainability Report: A Transformative Era of Action. Building a World that Works for Tomorrow.",\n    "type": "header",\n    "relevance_score": 0.95\n  },\n  {\n    "title": "Global R&D Investment",\n    "content": "GE inv'


PDFs: 100%|██████████| 33/33 [10:51<00:00, 19.73s/it]

RAW [Sysco%20FY24%20Sustainability%20Report]: '{\n  "chunks": [\n    {\n      "title": "Sysco Company Overview",\n      "content": "Sysco is the global leader in selling, marketing and distributing food products to restaurants, healthcare, and educational facilities. With over 76,000 colleagues and 340 distribution facilities worldwide, the company '
Done. Division: 32 | Extractions: 218 | Meta: 32


In [11]:
# Cell 8 — Save datasets
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

df_division   = pd.DataFrame(division_records)
df_extraction = pd.DataFrame(extraction_records)
df_meta       = pd.DataFrame(meta_records)

df_division.to_csv(DIVISION_DIR   / f"division_{timestamp}.csv",   index=False)
df_extraction.to_csv(EXTRACTION_DIR / f"extraction_{timestamp}.csv", index=False)
df_meta.to_csv(META_DIR           / f"meta_{timestamp}.csv",       index=False)

print("Saved:")
print(f"  division:   {len(df_division)} rows")
print(f"  extraction: {len(df_extraction)} rows")
print(f"  meta:       {len(df_meta)} rows")

Saved:
  division:   32 rows
  extraction: 218 rows
  meta:       32 rows
